In [ ]:
# WEEE model  

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from google.colab import drive
from sklearn.model_selection import LeaveOneGroupOut, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/WEEE_Dataset/dataset'
SAVE_PATH = '/content/drive/MyDrive/'

TRANSITION_BUFFER = 0.5
PHASE_BOUNDARIES  = [10.0, 20.0]

def get_phase(elapsed_min):
    for boundary in PHASE_BOUNDARIES:
        if abs(elapsed_min - boundary) <= TRANSITION_BUFFER:
            return None
    if   elapsed_min <= 10.0: return 'rest'
    elif elapsed_min <= 20.0: return 'cycling'
    else:                     return 'running'

def load_e4_acc(p_id):
    acc_path = os.path.join(DATA_PATH, p_id, 'E4', 'ACC.csv')
    if not os.path.exists(acc_path): return None
    raw      = pd.read_csv(acc_path, header=None)
    start_ts = float(raw.iloc[0, 0])
    hz       = float(raw.iloc[1, 0])
    df       = raw.iloc[2:].reset_index(drop=True).astype(float)
    df.columns = ['acc_x', 'acc_y', 'acc_z']
    df[['acc_x','acc_y','acc_z']] /= 64.0
    df['timestamp'] = pd.to_datetime(
        start_ts + np.arange(len(df)) / hz, unit='s')
    return df.set_index('timestamp')

def load_vo2(p_id):
    vo2_path = os.path.join(DATA_PATH, p_id, 'VO2', 'DataAverage.csv')
    df = pd.read_csv(vo2_path)
    df['timestamp'] = pd.to_datetime(df['Time'])
    return df.set_index('timestamp')

def extract_features(win):
    """Richer feature set — wrist ACC only."""
    mag = np.sqrt(win['acc_x']**2 + win['acc_y']**2 + win['acc_z']**2)
    mc  = mag.values - mag.values.mean()    
    feats = {
        
        'acc_mag_mean':   mag.mean(),
        'acc_mag_std':    mag.std(),
        'acc_mag_max':    mag.max(),
        'acc_mag_min':    mag.min(),
        'acc_mag_p2p':    mag.max() - mag.min(),
        'acc_mag_iqr':    np.percentile(mag, 75) - np.percentile(mag, 25),
        'acc_x_mean':     win['acc_x'].mean(),
        'acc_y_mean':     win['acc_y'].mean(),
        'acc_z_mean':     win['acc_z'].mean(),
        'acc_x_std':      win['acc_x'].std(),
        'acc_y_std':      win['acc_y'].std(),
        'acc_z_std':      win['acc_z'].std(),
         
        'acc_mag_energy': (mag**2).mean(),
        'acc_sma':        (win['acc_x'].abs() +
                           win['acc_y'].abs() +
                           win['acc_z'].abs()).mean(),
 
        'acc_mag_zcr':    ((mc[:-1] * mc[1:]) < 0).sum() / len(mag),
        
        'acc_xy_corr':    win['acc_x'].corr(win['acc_y']),
        'acc_xz_corr':    win['acc_x'].corr(win['acc_z']),
        'acc_yz_corr':    win['acc_y'].corr(win['acc_z']),
    }
    return feats

def process_all_participants():
    all_data = []
    participant_ids = [f"P{i:02d}" for i in range(1, 17)]
    for p_id in participant_ids:
        try:
            print(f"Processing {p_id}...")
            acc_df = load_e4_acc(p_id)
            if acc_df is None: continue
            vo2_df   = load_vo2(p_id)
            res_acc  = acc_df.resample('1S').mean()
            res_vo2  = vo2_df.resample('1S').interpolate(method='linear')
            merged   = pd.merge(res_acc, res_vo2,
                                left_index=True, right_index=True, how='inner')
            if merged.empty: continue
            merged['elapsed_min'] = (
                merged.index - merged.index[0]).total_seconds() / 60.0
            merged = merged[merged['VO2[mL/kg/min]'] > 1.0]

            window_size, step_size = 5, 2
            for i in range(0, len(merged) - window_size, step_size):
                win   = merged.iloc[i: i + window_size]
                phase = get_phase(win['elapsed_min'].iloc[-1])
                if phase is None: continue
                feats = extract_features(win)
                all_data.append({
                    'participant_id': p_id,
                    'phase':          phase,
                    'target_MET':     win['VO2[mL/kg/min]'].iloc[-1] / 3.5,
                    **feats,
                })
        except Exception as e:
            print(f"  Error {p_id}: {e}")
    return pd.DataFrame(all_data)

df_total = process_all_participants()
feature_cols = [c for c in df_total.columns
                if c not in ['participant_id','phase','target_MET']]

df_rest    = df_total[df_total['phase'] == 'rest'].copy()
df_cycling = df_total[df_total['phase'] == 'cycling'].copy()
df_running = df_total[df_total['phase'] == 'running'].copy()

print(f"\nWindows — rest: {len(df_rest):,} | "
      f"cycling: {len(df_cycling):,} | running: {len(df_running):,}")
print(f"\nMET stats:")
for phase, df in [('rest',df_rest),('cycling',df_cycling),('running',df_running)]:
    print(f"  {phase:8s}: mean={df['target_MET'].mean():.3f} "
          f"std={df['target_MET'].std():.3f} "
          f"range=[{df['target_MET'].min():.2f}, {df['target_MET'].max():.2f}]")


#LOPO
param_grid_rf = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2],
    'max_features':      ['sqrt', 'log2'],
}

logo = LeaveOneGroupOut()

def run_lopo_domain(df, domain_name, use_mean=False):
    """
    LOPO for one domain.
    use_mean=True → predict training mean (for rest domain).
    """
    X      = df[feature_cols]
    y      = df['target_MET']
    groups = df['participant_id']

    all_true, all_pred        = [], []
    fold_rmse, fold_r2, pids  = [], [], []
    print(f"\n--- LOPO: {domain_name} ---")

    for train_idx, test_idx in logo.split(X, y, groups=groups):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        held_out   = groups.iloc[test_idx].iloc[0]

        if use_mean:
            train_mean = y_tr.mean()
            y_pred     = np.full(len(y_te), train_mean)
        else:
            rf = RandomizedSearchCV(
                RandomForestRegressor(random_state=42),
                param_distributions=param_grid_rf,
                n_iter=15, cv=3,
                scoring='neg_root_mean_squared_error',
                random_state=42, n_jobs=-1)
            rf.fit(X_tr, y_tr)
            y_pred = rf.best_estimator_.predict(X_te)

        rmse = np.sqrt(mean_squared_error(y_te, y_pred))
        r2   = r2_score(y_te, y_pred)
        fold_rmse.append(rmse)
        fold_r2.append(r2)
        pids.append(held_out)
        all_true.extend(y_te.tolist())
        all_pred.extend(y_pred.tolist())
        print(f"  {held_out} | n={len(y_te):4d} | "
              f"RMSE={rmse:.4f} | R²={r2:.4f}")

    print(f"\n{domain_name}:")
    print(f"  RMSE : {np.mean(fold_rmse):.4f} ± {np.std(fold_rmse):.4f}")
    print(f"  R²   : {np.mean(fold_r2):.4f}  ± {np.std(fold_r2):.4f}")
    worst = pids[int(np.argmax(fold_rmse))]
    print(f"  Worst: {worst} (RMSE={max(fold_rmse):.4f})")
    return np.array(all_true), np.array(all_pred)

 
rest_true,    rest_pred    = run_lopo_domain(df_rest,    "Rest    (pop. mean)", use_mean=True)
cycling_true, cycling_pred = run_lopo_domain(df_cycling, "Cycling (RF)")
running_true, running_pred = run_lopo_domain(df_running, "Running (RF)")

all_true = np.concatenate([rest_true, cycling_true, running_true])
all_pred = np.concatenate([rest_pred, cycling_pred, running_pred])
print(f"\nOverall LOPO  RMSE : {np.sqrt(mean_squared_error(all_true, all_pred)):.4f}")
print(f"Overall LOPO  R²   : {r2_score(all_true, all_pred):.4f}")

# Running-only honest metrics
print(f"\nRunning-only  RMSE : "
      f"{np.sqrt(mean_squared_error(running_true, running_pred)):.4f}")
print(f"Running-only  R²   : {r2_score(running_true, running_pred):.4f}")
print(f"Cycling-only  RMSE : "
      f"{np.sqrt(mean_squared_error(cycling_true, cycling_pred)):.4f}")
print(f"Cycling-only  R²   : {r2_score(cycling_true, cycling_pred):.4f}")


 
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

def plot_scatter(ax, true, pred, title, color):
    ax.scatter(true, pred, alpha=0.3, color=color, edgecolors='w', s=25)
    lo, hi = min(true.min(), pred.min()), max(true.max(), pred.max())
    ax.plot([lo, hi], [lo, hi], '--k', lw=1.5)
    r2 = r2_score(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    ax.set_title(f'{title}\nR²={r2:.4f}  RMSE={rmse:.4f}', fontweight='bold')
    ax.set_xlabel('Actual MET')
    ax.set_ylabel('Predicted MET')

# Rest
axes[0].scatter(rest_true, rest_pred, alpha=0.2, color='#1f77b4', s=20)
lo, hi = rest_true.min(), rest_true.max()
axes[0].plot([lo, hi], [lo, hi], '--k', lw=1.5)
pop_mean = df_rest['target_MET'].mean()
axes[0].axhline(pop_mean, color='red', lw=2, linestyle='-.',
                label=f'Pop. mean={pop_mean:.2f}')
axes[0].set_title('Rest\nIMU uninformative → population mean', fontweight='bold')
axes[0].set_xlabel('Actual MET'); axes[0].set_ylabel('Predicted MET')
axes[0].legend(fontsize=9)

plot_scatter(axes[1], cycling_true, cycling_pred,
             'Cycling\n(wrist ACC — limited signal)', '#2ca02c')
plot_scatter(axes[2], running_true, running_pred,
             'Running\n(wrist ACC — better signal)', '#ff7f0e')
plot_scatter(axes[3], all_true, all_pred,
             'All Phases Combined', '#9467bd')

plt.suptitle('WEEE Model — LOPO CV (3-domain)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


 
print("\n--- Saving final models ---")

static_model = {
    'type':      'population_mean',
    'mean_MET':  df_rest['target_MET'].mean(),
    'std_MET':   df_rest['target_MET'].std(),
    'note':      'Wrist IMU uninformative at rest; use population mean ≈ 0.91 MET'
}

rf_cycling = RandomizedSearchCV(
    RandomForestRegressor(random_state=42), param_distributions=param_grid_rf,
    n_iter=15, cv=5, scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1)
rf_cycling.fit(df_cycling[feature_cols], df_cycling['target_MET'])

rf_running = RandomizedSearchCV(
    RandomForestRegressor(random_state=42), param_distributions=param_grid_rf,
    n_iter=15, cv=5, scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1)
rf_running.fit(df_running[feature_cols], df_running['target_MET'])

joblib.dump(static_model,
            os.path.join(SAVE_PATH, 'rf_static_final_no_hr.pkl'))
joblib.dump(rf_cycling.best_estimator_,
            os.path.join(SAVE_PATH, 'rf_cycling_final_no_hr.pkl'))
joblib.dump(rf_running.best_estimator_,
            os.path.join(SAVE_PATH, 'rf_running_final_no_hr.pkl'))

 
joblib.dump(rf_running.best_estimator_,
            os.path.join(SAVE_PATH, 'rf_dyn_final_no_hr.pkl'))

print(f"Rest    model saved (pop. mean = {static_model['mean_MET']:.3f})")
print(f"Cycling model saved")
print(f"Running model saved")
print(f"Dynamic model saved (= running model, for HAR pipeline compatibility)")


 
static_mdl  = joblib.load(os.path.join(SAVE_PATH, 'rf_static_final_no_hr.pkl'))
cycling_mdl = joblib.load(os.path.join(SAVE_PATH, 'rf_cycling_final_no_hr.pkl'))
running_mdl = joblib.load(os.path.join(SAVE_PATH, 'rf_running_final_no_hr.pkl'))

W_worker, L_load, T_window_min = 75.0, 10.0, 5.0

 
WEEE_DOMAIN = {0: 'rest', 1: 'rest', 2: 'running',
               3: 'running', 4: 'running', 5: 'running', 6: 'cycling'}

def predict_met(har_class, X_features):
    domain = WEEE_DOMAIN.get(har_class, 'running')
    if domain == 'rest':
        return static_mdl['mean_MET'], domain
    elif domain == 'cycling':
        return cycling_mdl.predict(X_features)[0], domain
    else:
        return running_mdl.predict(X_features)[0], domain

weee_feature_cols = [c for c in feature_cols
                     if c in ['acc_mag_mean','acc_mag_std','acc_mag_max',
                               'acc_mag_min','acc_mag_p2p','acc_mag_iqr',
                               'acc_x_mean','acc_y_mean','acc_z_mean',
                               'acc_x_std','acc_y_std','acc_z_std',
                               'acc_mag_energy','acc_sma','acc_mag_zcr',
                               'acc_xy_corr','acc_xz_corr','acc_yz_corr']]

X_example = pd.DataFrame({col: [0.0] for col in weee_feature_cols})
X_example['acc_mag_mean'] = 2.5
X_example['acc_mag_std']  = 0.8
X_example['acc_x_mean']   = 0.1
X_example['acc_y_mean']   = 0.9
X_example['acc_z_mean']   = 0.2

for har_class, name in {0:'Stand', 4:'Walk', 5:'Run', 6:'Bike'}.items():
    met, domain = predict_met(har_class, X_example[weee_feature_cols])
    energy_j    = met * (W_worker + L_load) * (T_window_min / 60) * 4184
    print(f"HAR={name:6s} | domain={domain:8s} | "
          f"MET={met:.3f} | Energy={energy_j:.1f}J ({energy_j/4184:.3f} kcal)")

In [ ]:
 
baseline = {
    'HAR': {
        'method':   'XGBoost + hand-crafted features',
        'val':      'LOSO (9 subjects)',
        'accuracy': 0.8762,
        'std':      0.0674,
        'features': feature_cols,   
    },
    'WEEE_rest': {
        'method':  'Population mean',
        'MET':      0.911,
        'RMSE':     0.2703,
    },
    'WEEE_cycling': {
        'method': 'RF + hand-crafted features',
        'val':    'LOPO (16 participants)',
        'RMSE':    1.2007,
        'R2':     -20.017,   
    },
    'WEEE_running': {
        'method': 'RF + hand-crafted features',
        'val':    'LOPO (16 participants)',
        
    },
}

import joblib, json
joblib.dump(baseline, '/content/drive/MyDrive/baseline_metrics.pkl')
print("Baseline frozen. Ready for SensorLM integration.")

Baseline frozen. Ready for SensorLM integration.


In [ ]:
 
import os
import glob
import warnings
import gc
import numpy as np
import pandas as pd
import scipy.fftpack
import joblib

from google.colab import drive
from sklearn.model_selection import LeaveOneGroupOut, train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

drive.mount('/content/drive')
BASE_PATH         = '/content/drive/MyDrive/HHAR'
ACTIVITY_PATH     = os.path.join(BASE_PATH, 'Activity recognition exp')
ANDROID_DATA_PATH = '/content/drive/MyDrive/Android_Data'
SAVE_PATH         = '/content/drive/MyDrive/'

 
LABEL_MAP = {
    
    'stand':      0, 'sit':        0, 'null':    -1,   # -1 = drop
    'bike':       6,
    'stairsup':   2, 'stairsdown': 3,
    'walk':       4,
   
    'elevator':   1,
    'running':    5,
    'walking':    4,
    'stairsup':   2,
    'stairsdown': 3,
}
 
ALL_CLASSES  = [0, 1, 2, 3, 4, 5, 6]
CLASS_NAMES  = {
    0: '대기/앉기(0)', 1: '엘리베이터(1)', 2: '계단UP(2)',
    3: '계단DOWN(3)', 4: '평지걷기(4)',   5: '뛰기(5)',  6: '자전거(6)'
}

def map_gt_hhar(gt_value):
    gt = str(gt_value).strip().lower()
    return LABEL_MAP.get(gt, -1)     

def map_folder_to_label(folder_name):
    """Parse Phyphox folder names like 'Walking_with_box', 'Elevator_without_box'."""
    name  = folder_name.lower()
    label = -1
    if   'elevator'   in name: label = 1
    elif 'stairsup'   in name: label = 2
    elif 'stairsdown' in name: label = 3
    elif 'walking'    in name: label = 4
    elif 'running'    in name: label = 5
    carrying_load = 'with_box' in name   # True / False
    return label, carrying_load


 
def extract_sensor_features(window_df, sensor_prefix, sample_hz=100.0):
    feats = {}
    x   = window_df['x'].values.astype(float)
    y   = window_df['y'].values.astype(float)
    z   = window_df['z'].values.astype(float)
    mag = np.sqrt(x*x + y*y + z*z)             
 
    if sensor_prefix == 'acc' and mag.mean() > 5.0:
        x = x - x.mean()
        y = y - y.mean()
        z = z - z.mean()
        mag = np.sqrt(x*x + y*y + z*z)

    mag_centered = mag - mag.mean()

   
    feats[f'{sensor_prefix}_x_mean']     = float(np.mean(x))
    feats[f'{sensor_prefix}_y_mean']     = float(np.mean(y))
    feats[f'{sensor_prefix}_z_mean']     = float(np.mean(z))
    feats[f'{sensor_prefix}_x_std']      = float(np.std(x))
    feats[f'{sensor_prefix}_y_std']      = float(np.std(y))
    feats[f'{sensor_prefix}_z_std']      = float(np.std(z))
    feats[f'{sensor_prefix}_mag_mean']   = float(np.mean(mag))
    feats[f'{sensor_prefix}_mag_std']    = float(np.std(mag))
    feats[f'{sensor_prefix}_mag_p2p']    = float(np.ptp(mag))
    feats[f'{sensor_prefix}_mag_energy'] = float(np.sum(mag**2) / len(mag))
    feats[f'{sensor_prefix}_sma']        = float(np.sum(np.abs(x)+np.abs(y)+np.abs(z)) / len(mag))
    feats[f'{sensor_prefix}_mag_75th']   = float(np.percentile(mag, 75))
    feats[f'{sensor_prefix}_mag_25th']   = float(np.percentile(mag, 25))

     
    feats[f'{sensor_prefix}_zcr'] = float(
        ((mag_centered[:-1] * mag_centered[1:]) < 0).sum() / len(mag)
    )

    
    dt = 1.0 / sample_hz
    feats[f'{sensor_prefix}_z_drift']    = float(np.sum(z) * dt)
    feats[f'{sensor_prefix}_z_drift_abs']= float(abs(np.sum(z) * dt))

    
    def _skew(a):
        m, s = a.mean(), a.std()
        return float(((a-m)**3).mean() / (s**3 + 1e-9))
    def _kurt(a):
        m, s = a.mean(), a.std()
        return float(((a-m)**4).mean() / (s**4 + 1e-9) - 3.0)
    feats[f'{sensor_prefix}_z_skew']     = _skew(z)
    feats[f'{sensor_prefix}_z_kurt']     = _kurt(z)
    feats[f'{sensor_prefix}_mag_skew']   = _skew(mag)
    feats[f'{sensor_prefix}_mag_kurt']   = _kurt(mag)

    
    z_pos = z[z > 0]
    z_neg = z[z < 0]
    feats[f'{sensor_prefix}_z_pos_mean'] = float(z_pos.mean()) if len(z_pos) else 0.0
    feats[f'{sensor_prefix}_z_neg_mean'] = float(z_neg.mean()) if len(z_neg) else 0.0
    feats[f'{sensor_prefix}_z_peak_asym']= (
        feats[f'{sensor_prefix}_z_pos_mean'] + feats[f'{sensor_prefix}_z_neg_mean']
    )    
    fft_vals  = np.abs(scipy.fftpack.fft(mag))
    freqs     = scipy.fftpack.fftfreq(len(mag), d=1.0/sample_hz)
    peak_idx  = 1 + np.argmax(fft_vals[1:]) if len(fft_vals) > 1 else 0
    fs        = fft_vals.sum() + 1e-9
    feats[f'{sensor_prefix}_dominant_freq']    = float(abs(freqs[peak_idx]))
    feats[f'{sensor_prefix}_spectral_entropy'] = float(
        -np.sum((fft_vals / fs) * np.log(fft_vals / fs + 1e-9))
    )
    
    fft_z = np.abs(scipy.fftpack.fft(z - z.mean()))
    z_pidx = 1 + np.argmax(fft_z[1:]) if len(fft_z) > 1 else 0
    feats[f'{sensor_prefix}_z_dominant_freq'] = float(abs(freqs[z_pidx]))

    return feats

def safe_read_csv(path):
    for enc in ['utf-8', 'latin1', 'cp1252']:
        try:    return pd.read_csv(path, encoding=enc)
        except: continue
    return None

def standardize_columns(df):
    return df.rename(columns={c: c.strip().lower() for c in df.columns})


 
print("--- Processing HHAR Data ---")

def infer_sensor(path):
    n = os.path.basename(path).lower()
    return 'gyro' if 'gyro' in n else 'acc' if 'acc' in n else 'unknown'

all_csv  = [f for f in glob.glob(os.path.join(ACTIVITY_PATH, '*.csv'))
            if 'readme' not in f.lower()]
 
files_df = pd.DataFrame({
    'file_path':   f,
    'sensor_type': infer_sensor(f),
    'is_phone':    'phone' in os.path.basename(f).lower(),
} for f in all_csv)
files_df = files_df[files_df['is_phone']].copy()

def load_raw_phone_csv(file_path):
    df = safe_read_csv(file_path)
    if df is None: return None
    df = standardize_columns(df)
    req = ['creation_time','x','y','z','user','model','device','gt']
    if not all(c in df.columns for c in req): return None
    df = df[req].dropna().copy()
    for col in ['creation_time','x','y','z']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna()
    df['class_label'] = df['gt'].apply(map_gt_hhar)
    df = df[df['class_label'] >= 0]           # drop null and unknowns
    df['time_sec'] = df['creation_time'] / 1e9
    return df.sort_values(['user','device','time_sec']).reset_index(drop=True)

acc_raw_path  = next((r.file_path for _,r in files_df.iterrows() if r.sensor_type=='acc'),  None)
gyro_raw_path = next((r.file_path for _,r in files_df.iterrows() if r.sensor_type=='gyro'), None)
acc_raw       = load_raw_phone_csv(acc_raw_path)
gyro_raw      = load_raw_phone_csv(gyro_raw_path)
print(f"HHAR acc  rows: {len(acc_raw):,} | classes: {sorted(acc_raw['class_label'].unique())}")
print(f"HHAR gyro rows: {len(gyro_raw):,} | classes: {sorted(gyro_raw['class_label'].unique())}")

def window_hhar(acc_raw, gyro_raw, window_sec=5.0, step_sec=2.5, min_samples=30):
    results = []
    for (user, device), acc_grp in acc_raw.groupby(['user','device']):
        gyro_grp = gyro_raw[(gyro_raw['user']==user) & (gyro_raw['device']==device)]
        if gyro_grp.empty: continue

        t_start = max(acc_grp['time_sec'].min(), gyro_grp['time_sec'].min())
        t_end   = min(acc_grp['time_sec'].max(), gyro_grp['time_sec'].max())
        if t_end - t_start < window_sec: continue

        acc_t, gyro_t = acc_grp['time_sec'].values, gyro_grp['time_sec'].values
        t, win_id = t_start, 0

        while t + window_sec <= t_end:
            a_win = acc_grp[(acc_t >= t) & (acc_t < t+window_sec)]
            g_win = gyro_grp[(gyro_t >= t) & (gyro_t < t+window_sec)]
            if len(a_win) >= min_samples and len(g_win) >= min_samples:
                label = int(a_win['class_label'].mode().iloc[0])
                results.append({
                    'class_label':    label,
                    'carrying_load':  0,          
                    'user':           user,
                    'device':         device,
                    'source':         'hhar',
                    'window_id':      win_id,
                    **extract_sensor_features(a_win, 'acc'),
                    **extract_sensor_features(g_win, 'gyro'),
                })
            t += step_sec
            win_id += 1
    return results

print("Windowing HHAR...")
hhar_windows = window_hhar(acc_raw, gyro_raw)
hhar_df      = pd.DataFrame(hhar_windows).fillna(0)
print(f"HHAR windows: {len(hhar_df):,}")
print(hhar_df['class_label'].value_counts().sort_index())
 


print("\n--- Processing Phyphox Data ---")

def load_phyphox_csv(path):
    if not os.path.exists(path): return None
    df = pd.read_csv(path)
    df.columns = ['time_sec','x','y','z'] + list(df.columns[4:])
    return df[['time_sec','x','y','z']].astype(np.float32)

def process_phyphox_folder(folder_path, label, carrying_load,
                            window_sec=5.0, step_sec=2.0, min_samples=50):
    acc_df  = load_phyphox_csv(os.path.join(folder_path, 'Linear Acceleration.csv'))
    gyro_df = load_phyphox_csv(os.path.join(folder_path, 'Gyroscope.csv'))
    if acc_df is None or gyro_df is None: return []

    max_time  = min(acc_df['time_sec'].max(), gyro_df['time_sec'].max())
    acc_t     = acc_df['time_sec'].values
    gyro_t    = gyro_df['time_sec'].values
    results, t = [], 0.0

    while t + window_sec <= max_time:
        a_win = acc_df.iloc[np.searchsorted(acc_t, t):np.searchsorted(acc_t, t+window_sec)]
        g_win = gyro_df.iloc[np.searchsorted(gyro_t,t):np.searchsorted(gyro_t,t+window_sec)]
        if len(a_win) >= min_samples and len(g_win) >= min_samples:
            results.append({
                'class_label':   label,
                'carrying_load': int(carrying_load),
                'user':          'phyphox_team',   
                'device':        'phyphox',
                'source':        'phyphox',
                'window_id':     int(t / step_sec),
                **extract_sensor_features(a_win, 'acc'),
                **extract_sensor_features(g_win, 'gyro'),
            })
        t += step_sec
    return results

all_phyphox = []
if os.path.exists(ANDROID_DATA_PATH):
    for folder_name in os.listdir(ANDROID_DATA_PATH):
        folder_path = os.path.join(ANDROID_DATA_PATH, folder_name)
        if not os.path.isdir(folder_path): continue
        label, carrying_load = map_folder_to_label(folder_name)
        if label < 0:
            print(f"  [SKIP] unrecognised folder: {folder_name}")
            continue
        n = len(process_phyphox_folder(folder_path, label, carrying_load))
        all_phyphox.extend(
            process_phyphox_folder(folder_path, label, carrying_load))
        print(f"  {folder_name:30s} → class={label}, load={carrying_load}, n={n}")

phyphox_df = pd.DataFrame(all_phyphox).fillna(0)
print(f"\nPhyphox windows: {len(phyphox_df):,}")
print(phyphox_df['class_label'].value_counts().sort_index())


 
combined_df  = pd.concat([hhar_df, phyphox_df], ignore_index=True)
meta_cols    = ['class_label','carrying_load','user','device','source','window_id']
feature_cols = [c for c in combined_df.columns if c not in meta_cols]
feature_cols = feature_cols + ['carrying_load']   # include load as a feature

print(f"\nCombined dataset: {len(combined_df):,} windows")
print(f"Class distribution:\n{combined_df['class_label'].value_counts().sort_index()}")
print(f"Features: {len(feature_cols)}")

NUM_CLASSES = len(combined_df['class_label'].unique())
print(f"Unique classes: {sorted(combined_df['class_label'].unique())}")


 
print("\n--- LOSO Cross-Validation (HHAR subjects) ---")

X_hhar      = hhar_df[feature_cols].copy()
y_hhar      = hhar_df['class_label'].astype(int).copy()
loso_groups = hhar_df['user'].astype(str)

logo            = LeaveOneGroupOut()
unique_subjects = loso_groups.unique()
print(f"Subjects: {sorted(unique_subjects)}  →  {len(unique_subjects)} folds")

all_y_true, all_y_pred = [], []
fold_accs, skipped     = [], []

def make_xgb(n_classes):
    return XGBClassifier(
        n_estimators     = 300,
        max_depth        = 6,
        learning_rate    = 0.1,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        min_child_weight = 3,
        reg_alpha        = 0.1,
        reg_lambda       = 1.0,
        objective        = 'multi:softprob',
        num_class        = n_classes,
        eval_metric      = 'mlogloss',
        random_state     = 42,
        n_jobs           = -1,
    )

for fold_i, (train_idx, test_idx) in enumerate(
        logo.split(X_hhar, y_hhar, groups=loso_groups)):

    subject  = loso_groups.iloc[test_idx].iloc[0]
    X_tr, X_te = X_hhar.iloc[train_idx], X_hhar.iloc[test_idx]
    y_tr, y_te = y_hhar.iloc[train_idx], y_hhar.iloc[test_idx]

    unseen = set(y_te.unique()) - set(y_tr.unique())
    if unseen or len(y_te) < 10:
        reason = f"unseen classes {unseen}" if unseen else "too few test samples"
        print(f"  [SKIP] {subject}: {reason}")
        skipped.append(subject)
        continue

 
    fold_classes  = sorted(y_tr.unique())
    lmap          = {o:n for n,o in enumerate(fold_classes)}
    inv_lmap      = {n:o for o,n in lmap.items()}
    y_tr_m        = y_tr.map(lmap)
    y_te_m        = y_te.map(lmap)

    model = make_xgb(len(fold_classes))
    model.fit(X_tr, y_tr_m, eval_set=[(X_te, y_te_m)], verbose=False)

    y_pred_orig = pd.Series(model.predict(X_te)).map(inv_lmap).values
    acc         = accuracy_score(y_te, y_pred_orig)
    fold_accs.append(acc)
    all_y_true.extend(y_te.tolist())
    all_y_pred.extend(y_pred_orig.tolist())
    print(f"  {subject} | classes={fold_classes} | n={len(y_te):4d} | Acc={acc:.4f}")

print(f"\nLOSO  ({len(fold_accs)} valid, {len(skipped)} skipped)")
if fold_accs:
    print(f"  Mean ± Std : {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}")
    print(f"  Min  / Max : {np.min(fold_accs):.4f} / {np.max(fold_accs):.4f}")

    present_labels = sorted(set(all_y_true))
    present_names  = [CLASS_NAMES[i] for i in present_labels]
    print("\nAggregated Classification Report (HHAR LOSO):")
    print(classification_report(all_y_true, all_y_pred,
                                labels=present_labels, target_names=present_names))


 
print("--- Retraining final model on HHAR + Phyphox ---")

X_final = combined_df[feature_cols].copy()
y_final = combined_df['class_label'].astype(int).copy()

final_model = make_xgb(len(y_final.unique()))
final_model.fit(X_final, y_final)

har_model_path = os.path.join(SAVE_PATH, 'har_master_courier_loso.pkl')
joblib.dump(final_model, har_model_path)
print(f"Model saved → {har_model_path}")

 
print("\nHAR (LOSO-validated) + WEEE (IMU-Only)")
try:
    har_router   = joblib.load(har_model_path)
    weee_static  = joblib.load(os.path.join(SAVE_PATH, 'rf_static_final_no_hr.pkl'))
    weee_dynamic = joblib.load(os.path.join(SAVE_PATH, 'rf_dyn_final_no_hr.pkl'))
    print("All models loaded")
except Exception as e:
    print(f"Load error: {e}")

W_worker, L_load, T_window_min = 75.0, 10.0, 5.0

 
_example_overrides = {
    'acc_x_mean': 0.1, 'acc_y_mean': 0.9, 'acc_z_mean': 0.2,
    'acc_mag_mean': 2.5, 'acc_mag_std': 0.8, 'acc_mag_p2p': 2.1,
    'acc_mag_energy': 15.3, 'acc_sma': 4.2,
    'acc_mag_75th': 3.0, 'acc_mag_25th': 1.8, 'acc_zcr': 0.02,
    'acc_dominant_freq': 1.8, 'acc_spectral_entropy': 3.1,
    'gyro_x_mean': 0.01, 'gyro_y_mean': 0.02, 'gyro_z_mean': 0.01,
    'gyro_mag_mean': 0.05, 'gyro_mag_std': 0.02, 'gyro_mag_p2p': 0.1,
    'gyro_mag_energy': 0.005, 'gyro_sma': 0.04,
    'gyro_mag_75th': 0.06, 'gyro_mag_25th': 0.03, 'gyro_zcr': 0.01,
    'gyro_dominant_freq': 1.8, 'gyro_spectral_entropy': 2.9,
    'carrying_load': 1,
 
    'acc_mag_max': 4.0, 'acc_mag_min': 0.5, 'acc_mag_iqr': 1.2,
    'acc_mag_zcr': 0.15, 'acc_xy_corr': 0.1, 'acc_xz_corr': -0.2, 'acc_yz_corr': 0.05,
}
expected_cols = list(har_router.feature_names_in_)
X_current = pd.DataFrame([{col: _example_overrides.get(col, 0.0) for col in expected_cols}])

X_current = X_current[har_router.feature_names_in_]
action_class = int(har_router.predict(X_current)[0])
print(f"[HAR] Detected: {CLASS_NAMES.get(action_class, 'Unknown')}")
print(f"[HAR] Carrying load: {'Yes' if X_current['carrying_load'].iloc[0] else 'No'}")
 
def _weee_input(model):
    cols = list(getattr(model, 'feature_names_in_', [])) or [
        'acc_mag_mean','acc_mag_std','acc_mag_max','acc_mag_min','acc_mag_p2p','acc_mag_iqr',
        'acc_x_mean','acc_y_mean','acc_z_mean','acc_x_std','acc_y_std','acc_z_std',
        'acc_mag_energy','acc_sma','acc_mag_zcr','acc_xy_corr','acc_xz_corr','acc_yz_corr']
    return pd.DataFrame([{c: _example_overrides.get(c, 0.0) for c in cols}])[cols]
static_acts   = [0, 1]     

 
if action_class in static_acts:
    if isinstance(weee_static, dict):
        predicted_met = float(weee_static.get('mean_MET', 1.3))
    else:
        predicted_met = float(weee_static.predict(_weee_input(weee_static))[0])
else:
    if isinstance(weee_dynamic, dict):
        predicted_met = float(weee_dynamic.get('mean_MET', 3.0))
    else:
        predicted_met = float(weee_dynamic.predict(_weee_input(weee_dynamic))[0])
predicted_met = max(predicted_met, 1.0)    

energy_j = predicted_met * (W_worker + 0.5 * L_load) * (T_window_min / 60) * 4184  # Cload = 1 + 0.5*L/W (Soule & Goldman 1969); matches server/app
print(f"MET: {predicted_met:.2f} | Energy: {energy_j:.1f} J ({energy_j/4184:.3f} kcal)")
print("-" * 50)